# BERT sur Tiny Shakespeare

Ce notebook utilise un petit BERT préentraîné pour du masked language modeling sur Tiny Shakespeare, avec un split train / validation / test complet et une recherche Optuna sur les hyperparamètres.

In [24]:
%pip install --quiet transformers datasets accelerate optuna matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import shutil
import random
import numpy as np
import math
import torch
from datasets import Dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# ====================== CONFIGURATION ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====================== CHEMINS ======================
NOTEBOOK_DIR = Path("/kaggle/working")
INPUT_DATA_PATH = Path("/kaggle/input/datasets/mlonjoansafagnibo/bertyuiosp/tiny_shakespeare.txt")

DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "bert_mini_tiny_shakespeare"
MODEL_DIR = NOTEBOOK_DIR / "models" / "bert_mini_tiny_shakespeare"

# Création des dossiers
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ====================== COPIE DU DATASET ======================
DATA_PATH = DATA_DIR / "tiny_shakespeare.txt"

if INPUT_DATA_PATH.exists():
    shutil.copy(INPUT_DATA_PATH, DATA_PATH)
    print("✅ Dataset copié avec succès vers working directory")
else:
    print("❌ Fichier non trouvé à l'emplacement input")

# ====================== PARAMÈTRES ======================
MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 64
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
MLM_PROBABILITY = 0.15

SEARCH_EPOCHS = 5
FINAL_EPOCHS = 3
N_TRIALS = 2

# ====================== AFFICHAGE ======================
print("CUDA available:", torch.cuda.is_available())
print("Data path (working):", DATA_PATH)
print("Model:", MODEL_NAME)

CUDA available: False
Data path: D:\100DaysofML\Notebooks\095_BERT\tiny_shakespeare_gpt\data\tiny_shakespeare.txt
Model: google/bert_uncased_L-4_H-256_A-4


In [26]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing Tiny Shakespeare file: {DATA_PATH}")

with DATA_PATH.open("r", encoding="utf-8") as handle:
    lines = [line.strip() for line in handle.readlines()]

lines = [line for line in lines if line]
print("Total non-empty lines:", len(lines))
print("Sample line:", lines[0])

raw_dataset = Dataset.from_dict({"text": lines})
split_80_20 = raw_dataset.train_test_split(test_size=0.20, seed=SEED)
train_raw = split_80_20["train"]
temp_raw = split_80_20["test"]
split_10_10 = temp_raw.train_test_split(test_size=0.50, seed=SEED)
val_raw = split_10_10["train"]
test_raw = split_10_10["test"]

print("Train rows:", len(train_raw))
print("Validation rows:", len(val_raw))
print("Test rows:", len(test_raw))

Total non-empty lines: 32777
Sample line: First Citizen:
Train rows: 26221
Validation rows: 3278
Test rows: 3278


In [27]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_tokenized = train_raw.map(tokenize_batch, batched=True, remove_columns=["text"])
val_tokenized = val_raw.map(tokenize_batch, batched=True, remove_columns=["text"])
test_tokenized = test_raw.map(tokenize_batch, batched=True, remove_columns=["text"])

print(train_tokenized[0])
print("Tokenizer vocab size:", tokenizer.vocab_size)

Map:   0%|          | 0/26221 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3278 [00:00<?, ? examples/s]

{'input_ids': [101, 8761, 1010, 5469, 1010, 3571, 1998, 19306, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [28]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=MLM_PROBABILITY)


def model_init():
    return AutoModelForMaskedLM.from_pretrained(MODEL_NAME, local_files_only=True)


training_args_search = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "optuna"),
    num_train_epochs=SEARCH_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

search_trainer = Trainer(
    model_init=model_init,
    args=training_args_search,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)


def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }


print("Ready for Optuna search")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready for Optuna search


In [ ]:
best_run = search_trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=N_TRIALS,
)

print("Best run:")
print(best_run)

best_learning_rate = best_run.hyperparameters["learning_rate"]
best_weight_decay = best_run.hyperparameters["weight_decay"]

final_training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "final"),
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=best_learning_rate,
    weight_decay=best_weight_decay,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

model = model_init()
trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()
val_results = trainer.evaluate()
test_results = trainer.evaluate(test_tokenized, metric_key_prefix="test")

val_loss = val_results["eval_loss"]
test_loss = test_results["test_loss"]
val_perplexity = math.exp(val_loss)
test_perplexity = math.exp(test_loss)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {val_perplexity:.2f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_perplexity:.2f}")

[I 2026-05-25 00:57:03,514] A new study created in memory with name: no-name-3e268180-eac6-4a2c-bed4-fd5067642f61


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
model = trainer.model
model.to(final_training_args.device)
model.eval()

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer, top_k=5)

prompts = [
    "To be, or not to be, that is the [MASK].",
    "The king said [MASK] to his noblemen.",
    "Tomorrow, and tomorrow, and tomorrow, [MASK] in this petty pace.",
]

for prompt in prompts:
    print("\nPROMPT:", prompt)
    predictions = fill_mask(prompt)
    for item in predictions:
        print(f"- {item['token_str'].strip()} | score={item['score']:.4f}")


PROMPT: To be, or not to be, that is the [MASK].
- randomly | score=0.0001
- demolished | score=0.0001
- ##sf | score=0.0001
- overseen | score=0.0001
- single | score=0.0001

PROMPT: The king said [MASK] to his noblemen.
- policies | score=0.0001
- swept | score=0.0001
- sail | score=0.0001
- ##ich | score=0.0001
- tourist | score=0.0001

PROMPT: Tomorrow, and tomorrow, and tomorrow, [MASK] in this petty pace.
- tying | score=0.0001
- hero | score=0.0001
- obstacles | score=0.0001
- [unused70] | score=0.0001
- ##analysis | score=0.0001
